In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    cross_val_predict,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

bookings = pd.read_csv("appointments_clean.csv")

features = [
    "age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
    "reminder_sent", "booking_channel", "visit_type",
]
X = bookings[features].copy()
X["reminder_sent"] = X["reminder_sent"].map({"No": 0, "Yes": 1})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"], dtype=int)
y = bookings["no_show"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline = y_test.eq(0).mean()
print("Features:", X.shape[1])
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Test no-shows:", y_test.sum())
print(f"Overall no-show rate: {y.mean():.1%}")
print(f"Always-attended baseline accuracy: {baseline:.4f}")

Features: 11
Training rows: 792
Test rows: 198
Test no-shows: 49
Overall no-show rate: 22.0%
Always-attended baseline accuracy: 0.7525


In [2]:
def evaluate_classifier(model, features, actual):
    predictions = model.predict(features)
    probabilities = model.predict_proba(features)[:, 1]
    return {
        "accuracy": accuracy_score(actual, predictions),
        "recall": recall_score(actual, predictions, zero_division=0),
        "roc_auc": roc_auc_score(actual, probabilities),
    }


# Task 3.2 — Logistic regression with training-only scaling.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logistic = LogisticRegression(max_iter=1000)
logistic.fit(X_train_s, y_train)

logistic_predictions = logistic.predict(X_test_s)
print("Logistic regression confusion matrix, labels [0, 1]:")
print(confusion_matrix(y_test, logistic_predictions, labels=[0, 1]))
print(f"Precision: {precision_score(y_test, logistic_predictions, zero_division=0):.3f}")
print(f"F1: {f1_score(y_test, logistic_predictions, zero_division=0):.3f}")

coefficients = pd.Series(logistic.coef_[0], index=X.columns, name="coefficient")
display(coefficients.to_frame().round(3))

reminder_coef = coefficients["reminder_sent"]
reminder_sd = scaler.scale_[X.columns.get_loc("reminder_sent")]
odds_per_sd = np.exp(reminder_coef)
odds_no_to_yes = np.exp(reminder_coef / reminder_sd)

print(f"Reminder odds ratio per one standard deviation: {odds_per_sd:.3f}")
print(f"Reminder odds ratio for No → Yes: {odds_no_to_yes:.3f}")
print("The ratios differ because a No-to-Yes change spans 1 / SD scaled units.")
print(
    f"In this fitted model, holding the other features constant, a reminder "
    f"is associated with {(1 - odds_no_to_yes):.1%} lower odds of a no-show."
)

# Task 3.3 — Compare the full tree and the shallow tree.
full_tree = DecisionTreeClassifier(random_state=42)
shallow_tree = DecisionTreeClassifier(max_depth=3, random_state=42)

for name, tree in [("Full tree", full_tree), ("Depth-3 tree", shallow_tree)]:
    tree.fit(X_train, y_train)
    train_accuracy = tree.score(X_train, y_train)
    test_accuracy = tree.score(X_test, y_test)
    print(
        f"{name} (depth {tree.get_depth()}): "
        f"train={train_accuracy:.3f}, test={test_accuracy:.3f}, "
        f"gap={train_accuracy - test_accuracy:.3f}"
    )

# Task 3.4 — Random forest and model comparison.
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)

print(f"Random forest training accuracy: {forest.score(X_train, y_train):.3f}")
importances = pd.Series(
    forest.feature_importances_, index=X.columns, name="importance"
)
display(importances.nlargest(3).to_frame().round(3))

comparison = pd.DataFrame.from_dict(
    {
        "Logistic regression": evaluate_classifier(logistic, X_test_s, y_test),
        "Depth-3 tree": evaluate_classifier(shallow_tree, X_test, y_test),
        "Random forest": evaluate_classifier(forest, X_test, y_test),
    },
    orient="index",
)
display(comparison.round(3))
print("Highest test ROC-AUC for reporting:", comparison["roc_auc"].idxmax())

Logistic regression confusion matrix, labels [0, 1]:
[[139  10]
 [ 32  17]]
Precision: 0.630
F1: 0.447


,coefficient
age,-0.254
distance_km,0.253
fee_inr,0.453
lead_days,0.614
prior_no_shows,0.650
reminder_sent,-0.581
booking_channel_App,-0.047
booking_channel_Phone,0.216
booking_channel_Walk-In,-0.186
visit_type_Consultation,0.016


Reminder odds ratio per one standard deviation: 0.560
Reminder odds ratio for No → Yes: 0.309
The ratios differ because a No-to-Yes change spans 1 / SD scaled units.
In this fitted model, holding the other features constant, a reminder is associated with 69.1% lower odds of a no-show.
Full tree (depth 15): train=1.000, test=0.672, gap=0.328
Depth-3 tree (depth 3): train=0.822, test=0.778, gap=0.044
Random forest training accuracy: 1.000


,importance
lead_days,0.220
age,0.197
distance_km,0.195


,accuracy,recall,roc_auc
Logistic regression,0.788,0.347,0.789
Depth-3 tree,0.778,0.224,0.700
Random forest,0.747,0.306,0.706


Highest test ROC-AUC for reporting: Logistic regression


In [3]:
search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
    cv=5,
    scoring="roc_auc",
)
search.fit(X_train, y_train)
print("Best forest settings:", search.best_params_)
print(f"Tuned forest training CV ROC-AUC: {search.best_score_:.4f}")

logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

candidates = {
    "Logistic regression": logistic_pipeline,
    "Depth-3 tree": shallow_tree,
    "Random forest": forest,
}

cv_scores = {}
for name, model in candidates.items():
    cv_scores[name] = cross_val_score(
        model, X_train, y_train, cv=5, scoring="roc_auc"
    ).mean()

candidates["Tuned random forest"] = search.best_estimator_
cv_scores["Tuned random forest"] = search.best_score_

cv_table = pd.Series(cv_scores, name="mean_training_cv_auc").sort_values(
    ascending=False
)
display(cv_table.to_frame().round(4))

winner_name = cv_table.idxmax()
winner = candidates[winner_name]

# Select the threshold from out-of-fold training predictions.
oof_probabilities = cross_val_predict(
    winner, X_train, y_train, cv=5, method="predict_proba"
)[:, 1]

threshold_results = []
for threshold in [0.50, 0.35]:
    flags = (oof_probabilities >= threshold).astype(int)
    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_train, flags, zero_division=0),
        "recall": recall_score(y_train, flags, zero_division=0),
        "f1": f1_score(y_train, flags, zero_division=0),
        "flagged": int(flags.sum()),
    })

threshold_table = pd.DataFrame(threshold_results).set_index("threshold")
display(threshold_table.round(3))
chosen_threshold = threshold_table["f1"].idxmax()

# Evaluate the selected model and threshold on the held-out test set.
winner.fit(X_train, y_train)
test_probabilities = winner.predict_proba(X_test)[:, 1]
test_flags = (test_probabilities >= chosen_threshold).astype(int)

print("Selected model:", winner_name)
print(f"Selected threshold: {chosen_threshold:.2f}")
print(f"Test precision: {precision_score(y_test, test_flags, zero_division=0):.3f}")
print(f"Test recall: {recall_score(y_test, test_flags, zero_division=0):.3f}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, test_probabilities):.3f}")
print("Test bookings flagged:", test_flags.sum())

risk_scores = pd.DataFrame({
    "appointment_id": bookings.loc[X_test.index, "appointment_id"].to_numpy(),
    "no_show": y_test.to_numpy(),
    "risk_prob": test_probabilities,
    "flag": test_flags,
})
risk_scores.to_csv("test_risk_scores.csv", index=False)
print("Exported shape:", risk_scores.shape)

Best forest settings: {'max_depth': 3, 'n_estimators': 200}
Tuned forest training CV ROC-AUC: 0.7673


,mean_training_cv_auc
Logistic regression,0.7737
Tuned random forest,0.7673
Random forest,0.7466
Depth-3 tree,0.6784


,precision,recall,f1,flagged
threshold,,,,
0.50,0.575,0.249,0.347,73
0.35,0.513,0.467,0.489,154


Selected model: Logistic regression
Selected threshold: 0.35
Test precision: 0.571
Test recall: 0.571
Test ROC-AUC: 0.789
Test bookings flagged: 49
Exported shape: (198, 4)
